In [1]:
import pandas as pd
import numpy as np

cols = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
train = pd.read_csv("data/train_FD001.txt", sep=r"\s+", header=None, names=cols)
test = pd.read_csv("data/test_FD001.txt", sep=r"\s+", header=None, names=cols)
rul_test = pd.read_csv("data/RUL_FD001.txt", header=None, names=["RUL"])

print(train.shape, test.shape, rul_test.shape)

(20631, 26) (13096, 26) (100, 1)


In [2]:
max_cycle = train.groupby("unit")["cycle"].transform("max")
train["RUL"] = max_cycle - train["cycle"]
print(train[["unit", "cycle", "RUL"]].head())
print(train[["unit", "cycle", "RUL"]].tail())

   unit  cycle  RUL
0     1      1  191
1     1      2  190
2     1      3  189
3     1      4  188
4     1      5  187
       unit  cycle  RUL
20626   100    196    4
20627   100    197    3
20628   100    198    2
20629   100    199    1
20630   100    200    0


In [3]:
train["RUL"] = train["RUL"].clip(upper=125)
print(train["RUL"].describe())

count    20631.000000
mean        86.829286
std         41.673699
min          0.000000
25%         51.000000
50%        103.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64


In [4]:
features = ["op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
std = train[features].std()
useful = std[std > 1e-6].index.tolist()
print("Supprimés :", [c for c in features if c not in useful])
print("Gardés :", useful)

Supprimés : ['op3', 's1', 's5', 's10', 's16', 's18', 's19']
Gardés : ['op1', 'op2', 's2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train[useful] = scaler.fit_transform(train[useful])
test[useful] = scaler.transform(test[useful])

print(train[useful].mean().round(2).head())
print(train[useful].std().round(2).head())

op1    0.0
op2   -0.0
s2     0.0
s3    -0.0
s4     0.0
dtype: float64
op1    1.0
op2    1.0
s2     1.0
s3     1.0
s4     1.0
dtype: float64


In [6]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, val_idx = next(gss.split(train, groups=train["unit"]))
X_tr, y_tr = train.iloc[tr_idx][useful], train.iloc[tr_idx]["RUL"]
X_val, y_val = train.iloc[val_idx][useful], train.iloc[val_idx]["RUL"]
print(X_tr.shape, X_val.shape)
print(train.iloc[tr_idx]["unit"].nunique(), train.iloc[val_idx]["unit"].nunique())

(16561, 17) (4070, 17)
80 20


In [7]:
X_test = test.groupby("unit").last()[useful]
y_test = rul_test["RUL"].clip(upper=125)
print(X_test.shape, y_test.shape)

(100, 17) (100,)
